In [1]:
import torch
import torch.nn as nn

In [ ]:
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
                   [6.0, 5.0, 4.0, 3.0, 2.0, 1.0],
                   [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]]])

dim_in = x.shape[1]

dim_out = 2

batch = torch.stack((x, x))

dropout = nn.Dropout(0.5)

print(x.shape)

In [ ]:
class CausalAttention(nn.Module):

    def __init__ (self, dim_in, dim_out, context_len, dropout, qkv_bias = False):

        super().__init__()
        self.dim_out = dim_out
        self.W_query = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.W_key = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.W_value = nn.Linear(dim_in, dim_out, bias = qkv_bias)
        self.Dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_len, context_len), diagonal=1))

    def forward(self, x):
        batch_dim, num_tokens, dim_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim = -1)
        attn_weights = self.Dropout(attn_weights)

        context_vect = attn_weights @ values
        return context_vect

class MultiHeadAttentionWrapper(nn.Module):

    def __init__ (self, dim_in, dim_out, context_len, dropout, num_heads, qkv_bias = False):

        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(dim_in, dim_out, context_len, dropout, qkv_bias) for _ in range(num_heads)])

    def forward (self, x):
        return torch.cat([head(x) for head in self.heads], dim = -1)